In [1]:

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import logsumexp, softmax
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, log_loss

PROJECT_DIR = Path.cwd().parent
DATA_FILE = PROJECT_DIR / "data" / "processed" / "lpmc.csv"

df = pd.read_csv(DATA_FILE)

print("Observations:", len(df))
print("Variables:", len(df.columns))


Observations: 81086
Variables: 32


In [2]:

# Travel time matrix (hours)
T = np.column_stack([
    df["dur_walking"],
    df["dur_cycling"],
    (
        df["dur_pt_access"]
        + df["dur_pt_rail"]
        + df["dur_pt_bus"]
        + df["dur_pt_int"]
    ),
    df["dur_driving"]
])

# Travel cost matrix (GBP)
C = np.column_stack([
    np.zeros(len(df)),
    np.zeros(len(df)),
    df["cost_transit"],
    (
        df["cost_driving_fuel"]
        + df["cost_driving_ccharge"]
    )
])

# Observed mode choices: 0=Walk, 1=Cycle, 2=PT, 3=Car
y = df["travel_mode"].to_numpy(dtype=int) - 1

# Individual/household characteristics
licence = df["driving_license"].to_numpy(dtype=float)

ownership_1 = (
    df["car_ownership"] == 1
).to_numpy(dtype=float)

ownership_2 = (
    df["car_ownership"] == 2
).to_numpy(dtype=float)

print("T:", T.shape)
print("C:", C.shape)
print("y:", y.shape)


T: (81086, 4)
C: (81086, 4)
y: (81086,)


In [3]:

# Person identifier
person_id = (
    df["household_id"].astype(str)
    + "_"
    + df["person_n"].astype(str)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(T, y, groups=person_id)
)

print("Training trips:", len(train_idx))
print("Test trips:", len(test_idx))

print(
    "Overlapping persons:",
    len(
        set(person_id.iloc[train_idx])
        & set(person_id.iloc[test_idx])
    )
)


Training trips: 64982
Test trips: 16104
Overlapping persons: 0


In [6]:

def extended_mnl_nll(params, idx):

    (
        asc_cycle,
        asc_pt,
        asc_car,
        b_time,
        b_cost,
        b_licence,
        b_ownership_1,
        b_ownership_2
    ) = params

    # Alternative-specific constants
    ASC = np.array([
        0.0,
        asc_cycle,
        asc_pt,
        asc_car
    ])

    # Base utility
    V = (
        ASC
        + b_time * T[idx]
        + b_cost * C[idx]
    )

    # Add individual characteristics to car utility
    V[:, 3] += (
        b_licence * licence[idx]
        + b_ownership_1 * ownership_1[idx]
        + b_ownership_2 * ownership_2[idx]
    )

    # Log-likelihood
    log_p = (
        V[np.arange(len(idx)), y[idx]]
        - logsumexp(V, axis=1)
    )

    return -log_p.sum()



In [7]:

parameter_names = [
    "ASC_Cycle",
    "ASC_PT",
    "ASC_Car",
    "B_Time",
    "B_Cost",
    "B_Licence",
    "B_Ownership_1",
    "B_Ownership_2"
]

extended_model = minimize(
    fun=extended_mnl_nll,
    x0=np.zeros(8),
    args=(train_idx,),
    method="L-BFGS-B",
    options={"maxiter": 1000}
)

print("Converged:", extended_model.success)
print("Training Log-Likelihood:", -extended_model.fun)

results = pd.DataFrame({
    "Parameter": parameter_names,
    "Estimate": extended_model.x
})

display(results.round(5))


Converged: True
Training Log-Likelihood: -51041.51501950121


,Parameter,Estimate
0,ASC_Cycle,-3.97505
1,ASC_PT,-0.59424
2,ASC_Car,-4.14107
3,B_Time,-5.66645
4,B_Cost,-0.18215
5,B_Licence,0.70825
6,B_Ownership_1,2.69020
7,B_Ownership_2,3.36303


In [8]:

def predict_extended_mnl(params, idx):
    (
        asc_cycle,
        asc_pt,
        asc_car,
        b_time,
        b_cost,
        b_licence,
        b_ownership_1,
        b_ownership_2
    ) = params

    ASC = np.array([0.0, asc_cycle, asc_pt, asc_car])

    V = ASC + b_time * T[idx] + b_cost * C[idx]

    V[:, 3] += (
        b_licence * licence[idx]
        + b_ownership_1 * ownership_1[idx]
        + b_ownership_2 * ownership_2[idx]
    )

    return softmax(V, axis=1)


P_extended_test = predict_extended_mnl(
    extended_model.x, test_idx
)

y_test = y[test_idx]
y_pred_extended = P_extended_test.argmax(axis=1)

extended_accuracy = accuracy_score(
    y_test, y_pred_extended
)

extended_logloss = log_loss(
    y_test, P_extended_test, labels=[0, 1, 2, 3]
)

print(f"Extended MNL Test Accuracy: {extended_accuracy:.4f}")
print(f"Extended MNL Test Log Loss: {extended_logloss:.4f}")


Extended MNL Test Accuracy: 0.6999
Extended MNL Test Log Loss: 0.8121


In [9]:

def baseline_mnl_nll(params, idx):
    asc_cycle, asc_pt, asc_car, b_time, b_cost = params

    ASC = np.array([0.0, asc_cycle, asc_pt, asc_car])

    V = ASC + b_time * T[idx] + b_cost * C[idx]

    log_p = (
        V[np.arange(len(idx)), y[idx]]
        - logsumexp(V, axis=1)
    )

    return -log_p.sum()


baseline_model = minimize(
    baseline_mnl_nll,
    x0=np.zeros(5),
    args=(train_idx,),
    method="L-BFGS-B",
    options={"maxiter": 1000}
)

def predict_baseline_mnl(params, idx):
    asc_cycle, asc_pt, asc_car, b_time, b_cost = params

    ASC = np.array([0.0, asc_cycle, asc_pt, asc_car])

    V = ASC + b_time * T[idx] + b_cost * C[idx]

    return softmax(V, axis=1)


P_baseline_test = predict_baseline_mnl(
    baseline_model.x, test_idx
)

baseline_accuracy = accuracy_score(
    y_test, P_baseline_test.argmax(axis=1)
)

baseline_logloss = log_loss(
    y_test, P_baseline_test, labels=[0, 1, 2, 3]
)

comparison = pd.DataFrame({
    "Model": ["Baseline MNL", "Extended MNL"],
    "Parameters": [5, 8],
    "Train Log-Likelihood": [
        -baseline_model.fun,
        -extended_model.fun
    ],
    "Test Accuracy": [
        baseline_accuracy,
        extended_accuracy
    ],
    "Test Log Loss": [
        baseline_logloss,
        extended_logloss
    ]
})

display(comparison.round(4))


,Model,Parameters,Train Log-Likelihood,Test Accuracy,Test Log Loss
0,Baseline MNL,5,-59830.8143,0.5841,0.9406
1,Extended MNL,8,-51041.5150,0.6999,0.8121


In [10]:

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

mode_names = [
    "Walk",
    "Cycle",
    "Public Transport",
    "Car"
]

# Predictions from both models
y_pred_baseline = P_baseline_test.argmax(axis=1)
y_pred_extended = P_extended_test.argmax(axis=1)

# Confusion matrices
cm_baseline = confusion_matrix(
    y_test, y_pred_baseline, labels=[0, 1, 2, 3]
)

cm_extended = confusion_matrix(
    y_test, y_pred_extended, labels=[0, 1, 2, 3]
)

print("Baseline MNL Confusion Matrix:")
display(pd.DataFrame(
    cm_baseline,
    index=mode_names,
    columns=mode_names
))

print("Extended MNL Confusion Matrix:")
display(pd.DataFrame(
    cm_extended,
    index=mode_names,
    columns=mode_names
))

print("Extended MNL Classification Report:")
print(classification_report(
    y_test,
    y_pred_extended,
    labels=[0, 1, 2, 3],
    target_names=mode_names,
    zero_division=0
))


Baseline MNL Confusion Matrix:


,Walk,Cycle,Public Transport,Car
Walk,1998,0,341,471
Cycle,100,0,149,248
Public Transport,370,0,2645,2809
Car,1032,0,1177,4764


Extended MNL Confusion Matrix:


,Walk,Cycle,Public Transport,Car
Walk,1751,0,280,779
Cycle,86,0,196,215
Public Transport,436,2,3593,1793
Car,434,0,612,5927


Extended MNL Classification Report:
                  precision    recall  f1-score   support

            Walk       0.65      0.62      0.63      2810
           Cycle       0.00      0.00      0.00       497
Public Transport       0.77      0.62      0.68      5824
             Car       0.68      0.85      0.76      6973

        accuracy                           0.70     16104
       macro avg       0.52      0.52      0.52     16104
    weighted avg       0.68      0.70      0.69     16104



In [11]:

# Probability assigned to the actual chosen mode
chosen_prob_baseline = P_baseline_test[
    np.arange(len(y_test)), y_test
]

chosen_prob_extended = P_extended_test[
    np.arange(len(y_test)), y_test
]

# Average chosen-mode probability by actual mode
probability_summary = []

for i, mode in enumerate(mode_names):
    mask = y_test == i

    probability_summary.append({
        "Mode": mode,
        "Trips": mask.sum(),
        "Baseline Avg. Probability": (
            chosen_prob_baseline[mask].mean()
        ),
        "Extended Avg. Probability": (
            chosen_prob_extended[mask].mean()
        )
    })

probability_summary = pd.DataFrame(probability_summary)

display(probability_summary.round(4))


,Mode,Trips,Baseline Avg. Probability,Extended Avg. Probability
0,Walk,2810,0.4038,0.4353
1,Cycle,497,0.0374,0.0448
2,Public Transport,5824,0.4553,0.5472
3,Car,6973,0.5273,0.6441


In [13]:

from scipy.stats import norm

# Use training data only
idx = train_idx
n = len(idx)
K = len(extended_model.x)

# Design matrix: observations x alternatives x parameters
X = np.zeros((n, 4, K))

# Alternative-specific constants
X[:, 1, 0] = 1
X[:, 2, 1] = 1
X[:, 3, 2] = 1

# Time and cost
X[:, :, 3] = T[idx]
X[:, :, 4] = C[idx]

# Characteristics entering car utility
X[:, 3, 5] = licence[idx]
X[:, 3, 6] = ownership_1[idx]
X[:, 3, 7] = ownership_2[idx]

# Fitted probabilities
P_train = predict_extended_mnl(
    extended_model.x, idx
)

# Expected feature vector
expected_X = np.einsum("ni,nik->nk", P_train, X)

# Score contribution for each observation
chosen_X = X[np.arange(n), y[idx]]
scores = chosen_X - expected_X

# Hessian of negative log-likelihood
H = np.zeros((K, K))

for j in range(4):
    differences = X[:, j, :] - expected_X
    H += np.einsum(
        "n,ni,nj->ij",
        P_train[:, j],
        differences,
        differences
    )

# Conventional covariance
H_inv = np.linalg.inv(H)
standard_se = np.sqrt(np.diag(H_inv))

# Aggregate scores by person
train_person_id = person_id.iloc[idx].to_numpy()

score_df = pd.DataFrame(scores)
score_df["person_id"] = train_person_id

cluster_scores = (
    score_df.groupby("person_id")
    .sum()
    .to_numpy()
)

# Cluster-robust covariance
G = len(cluster_scores)
meat = cluster_scores.T @ cluster_scores

cluster_cov = H_inv @ meat @ H_inv

# Finite-sample correction
cluster_cov *= (
    G / (G - 1)
    * (n - 1) / (n - K)
)

cluster_se = np.sqrt(np.diag(cluster_cov))
cluster_z = extended_model.x / cluster_se
cluster_p = 2 * norm.sf(np.abs(cluster_z))

inference_table = pd.DataFrame({
    "Parameter": parameter_names,
    "Estimate": extended_model.x,
    "Standard SE": standard_se,
    "Cluster SE": cluster_se,
    "Cluster z": cluster_z,
    "p-value": cluster_p
})

display(inference_table.round(4))


,Parameter,Estimate,Standard SE,Cluster SE,Cluster z,p-value
0,ASC_Cycle,-3.9751,0.0297,0.0458,-86.7762,0.0
1,ASC_PT,-0.5942,0.0153,0.0212,-28.0603,0.0
2,ASC_Car,-4.1411,0.0402,0.0650,-63.7031,0.0
3,B_Time,-5.6665,0.0546,0.0772,-73.3542,0.0
4,B_Cost,-0.1821,0.0041,0.0055,-33.2740,0.0
5,B_Licence,0.7083,0.0224,0.0352,20.1332,0.0
6,B_Ownership_1,2.6902,0.0340,0.0538,49.9919,0.0
7,B_Ownership_2,3.3630,0.0366,0.0570,58.9672,0.0


In [14]:

# Baseline test predictions
P_base = predict_baseline_mnl(
    baseline_model.x, test_idx
)

P_ext = predict_extended_mnl(
    extended_model.x, test_idx
)

# Save original matrices
T_original = T.copy()
C_original = C.copy()

# Scenario A: PT fare +10%
C = C_original.copy()
C[:, 2] *= 1.10

P_base_fare = predict_baseline_mnl(
    baseline_model.x, test_idx
)
P_ext_fare = predict_extended_mnl(
    extended_model.x, test_idx
)

# Restore costs
C = C_original.copy()

# Scenario B: PT time -10%
T = T_original.copy()
T[:, 2] *= 0.90

P_base_time = predict_baseline_mnl(
    baseline_model.x, test_idx
)
P_ext_time = predict_extended_mnl(
    extended_model.x, test_idx
)

# Restore original data
T = T_original
C = C_original

# Compare aggregate PT shares
def pt_share(P):
    return P[:, 2].mean() * 100

comparison_policy = pd.DataFrame({
    "Model": ["Baseline MNL", "Extended MNL"],
    "Baseline PT (%)": [
        pt_share(P_base),
        pt_share(P_ext)
    ],
    "PT Fare +10% (%)": [
        pt_share(P_base_fare),
        pt_share(P_ext_fare)
    ],
    "PT Time -10% (%)": [
        pt_share(P_base_time),
        pt_share(P_ext_time)
    ]
})

comparison_policy["Fare Effect (pp)"] = (
    comparison_policy["PT Fare +10% (%)"]
    - comparison_policy["Baseline PT (%)"]
)

comparison_policy["Time Effect (pp)"] = (
    comparison_policy["PT Time -10% (%)"]
    - comparison_policy["Baseline PT (%)"]
)

display(comparison_policy.round(4))


,Model,Baseline PT (%),PT Fare +10% (%),PT Time -10% (%),Fare Effect (pp),Time Effect (pp)
0,Baseline MNL,35.4300,34.9688,39.8448,-0.4612,4.4148
1,Extended MNL,35.7229,35.3122,39.6102,-0.4107,3.8873


In [15]:

mode_names = [
    "Walk",
    "Cycle",
    "Public Transport",
    "Car"
]

def make_share_table(P_base, P_scenario, scenario_name):
    baseline = P_base.mean(axis=0) * 100
    scenario = P_scenario.mean(axis=0) * 100

    return pd.DataFrame({
        "Mode": mode_names,
        "Baseline (%)": baseline,
        scenario_name: scenario,
        "Change (pp)": scenario - baseline
    })

print("Extended MNL: PT Fare +10%")
display(
    make_share_table(
        P_ext,
        P_ext_fare,
        "PT Fare +10% (%)"
    ).round(4)
)

print("Extended MNL: PT Time -10%")
display(
    make_share_table(
        P_ext,
        P_ext_time,
        "PT Time -10% (%)"
    ).round(4)
)


Extended MNL: PT Fare +10%


,Mode,Baseline (%),PT Fare +10% (%),Change (pp)
0,Walk,17.7653,17.8726,0.1073
1,Cycle,3.0129,3.0584,0.0455
2,Public Transport,35.7229,35.3122,-0.4107
3,Car,43.4990,43.7569,0.2579


Extended MNL: PT Time -10%


,Mode,Baseline (%),PT Time -10% (%),Change (pp)
0,Walk,17.7653,16.9301,-0.8351
1,Cycle,3.0129,2.6181,-0.3948
2,Public Transport,35.7229,39.6102,3.8873
3,Car,43.4990,40.8415,-2.6574


## Summary of Findings

An Extended Multinomial Logit (MNL) model was estimated using travel time, travel cost, driving licence status, and household car ownership.

The extended specification improved out-of-sample predictive performance compared with the baseline MNL model. Test accuracy increased from 58.41% to 69.99%, while test log loss decreased from 0.9406 to 0.8121.

Driving licence status and household car ownership were positively associated with the relative utility of car travel. All eight estimated parameters remained statistically significant when standard errors were clustered at the individual level.

Policy sensitivity analysis showed that a 10% increase in public transport fares reduced the predicted PT mode share by 0.411 percentage points. A 10% reduction in PT travel time increased the predicted PT share by 3.887 percentage points.

Compared with the baseline MNL, the extended specification predicted smaller aggregate responses to both policy changes. These differences demonstrate how the inclusion of individual and household characteristics can affect not only predictive performance but also the results of travel demand scenario analysis.

The results provide a foundation for developing more flexible travel choice specifications and examining behavioural heterogeneity.